# 💎 YOLOSaphire — Complete Training & Analysis Notebook
### Farhan Ferdous · 2026 · Solar Panel Hotspot Detection
---
**Run cells in order. All results auto-saved. ZIP download at the end.**

| Cell | Purpose |
|------|---------|
| 1 | Install & GPU check |
| 2 | Download dataset (Roboflow) |
| 3 | Dataset analysis & charts |
| 4 | Train YOLOSaphire-Lite |
| 5 | Training curves & loss graphs |
| 6 | mAP, Precision & Recall charts |
| 7 | Confusion matrix heatmap |
| 8 | Predictions on first 10 images |
| 9 | CSABlock attention heatmaps |
| 10 | YOLO26 vs YOLOSaphire comparison |
| 11 | Excel export (full training data) |
| 12 | Download everything as ZIP |


## Cell 1 — Install & GPU Check

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 1 — Install & GPU Check               │
# └─────────────────────────────────────────────┘

!pip install -q torch torchvision pyyaml pillow roboflow openpyxl seaborn matplotlib tqdm

import torch, torchvision, sys, os

print("=" * 55)
print("  YOLOSaphire Environment Check")
print("=" * 55)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  PyTorch     : {torch.__version__}")
print(f"  Torchvision : {torchvision.__version__}")
print(f"  CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU         : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("=" * 55)

for d in ['runs/train', 'runs/predict', 'runs/analysis', 'runs/excel']:
    os.makedirs(d, exist_ok=True)
print("  Output dirs created ✓")
print("  ⚠️  Make sure GPU is ON: Runtime → Change runtime type → T4 GPU")

## Cell 2 — Download Dataset

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 2 — Download Dataset (Roboflow)        │
# └─────────────────────────────────────────────┘

from roboflow import Roboflow
import yaml
from pathlib import Path

rf = Roboflow(api_key="pvvn2hB9d6Oug8uOnbR2")
project = rf.workspace("american-university-of-sharjah").project("hotspot-detection-in-solar-panel")
version = project.version(3)
dataset = version.download("yolov8")

data_root = Path(dataset.location)
DATA_YAML = str(list(data_root.glob('*.yaml'))[0])
DATASET_PATH = str(data_root)

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

NUM_CLASSES = cfg['nc']
CLASS_NAMES = cfg.get('names', [str(i) for i in range(NUM_CLASSES)])

def count_imgs(split):
    for s in [split, 'valid' if split=='val' else split]:
        p = data_root / s / 'images'
        if p.exists(): return len(list(p.glob('*')))
    return 0

print(f"  Dataset      : {dataset.location}")
print(f"  Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"  Train imgs   : {count_imgs('train')}")
print(f"  Val imgs     : {count_imgs('val')}")
print(f"  data.yaml    : {DATA_YAML}")

## Cell 3 — Dataset Analysis

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 3 — Dataset Analysis & Visualisation  │
# └─────────────────────────────────────────────┘

import matplotlib.pyplot as plt
import numpy as np
import yaml
from pathlib import Path
from collections import Counter

with open(DATA_YAML) as f: cfg = yaml.safe_load(f)
class_names = cfg.get('names', [str(i) for i in range(cfg['nc'])])
num_classes = cfg['nc']
data_root   = Path(DATASET_PATH)

COLORS = ['#3B82F6','#F59E0B','#10B981','#EF4444','#8B5CF6','#EC4899','#14B8A6','#F97316']

def count_labels(split):
    for s in [split, 'valid' if split=='val' else split]:
        ld = data_root / s / 'labels'
        if ld.exists():
            counts = Counter(); boxes = []
            for f in ld.glob('*.txt'):
                for line in f.read_text().strip().split('\n'):
                    if line:
                        parts = line.split()
                        counts[int(parts[0])] += 1
                        if len(parts)==5: boxes.append([float(x) for x in parts[1:]])
            return counts, boxes
    return Counter(), []

train_counts, train_boxes = count_labels('train')
val_counts,   val_boxes   = count_labels('val')

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor('#0f172a')
for ax in axes.flat:
    ax.set_facecolor('#1e293b')
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values(): spine.set_color('#334155')

# 1. Class distribution
ax = axes[0,0]
x = np.arange(num_classes); w = 0.38
b1 = ax.bar(x-w/2,[train_counts.get(i,0) for i in range(num_classes)],w,color=COLORS[0],alpha=0.9,label='Train')
b2 = ax.bar(x+w/2,[val_counts.get(i,0)   for i in range(num_classes)],w,color=COLORS[1],alpha=0.9,label='Val')
ax.set_xticks(x); ax.set_xticklabels(class_names,color='#e2e8f0',fontsize=9)
ax.set_title('Class Distribution',color='#f1f5f9',fontsize=12,pad=10)
ax.set_ylabel('Count',color='#94a3b8')
ax.legend(facecolor='#0f172a',edgecolor='#334155',labelcolor='#e2e8f0')
for bar in [*b1,*b2]:
    h=bar.get_height()
    if h>0: ax.annotate(str(int(h)),xy=(bar.get_x()+bar.get_width()/2,h),xytext=(0,4),textcoords='offset points',ha='center',va='bottom',color='#e2e8f0',fontsize=8)

# 2. Pie
ax2=axes[0,1]
tot_t=sum(train_counts.values()); tot_v=sum(val_counts.values())
if tot_t+tot_v>0:
    wedges,texts,autotexts=ax2.pie([tot_t,tot_v],labels=['Train','Val'],autopct='%1.1f%%',colors=[COLORS[0],COLORS[2]],startangle=90,wedgeprops=dict(edgecolor='#0f172a',linewidth=2))
    for t in texts: t.set_color('#e2e8f0')
    for t in autotexts: t.set_color('#0f172a'); t.set_fontweight('bold')
ax2.set_title('Train / Val Split',color='#f1f5f9',fontsize=12,pad=10)

# 3. Box scatter
ax3=axes[0,2]
if train_boxes:
    bw=[b[2] for b in train_boxes]; bh=[b[3] for b in train_boxes]
    ax3.scatter(bw,bh,alpha=0.4,s=15,c=COLORS[4],edgecolors='none')
ax3.set_xlabel('Box Width (norm)',color='#94a3b8'); ax3.set_ylabel('Box Height (norm)',color='#94a3b8')
ax3.set_title('Bounding Box Size Distribution',color='#f1f5f9',fontsize=12,pad=10)
ax3.set_xlim(0,1); ax3.set_ylim(0,1)

# 4. Aspect ratio
ax4=axes[1,0]
if train_boxes:
    ratios=[b[2]/(b[3]+1e-6) for b in train_boxes]
    ax4.hist(ratios,bins=30,color=COLORS[3],alpha=0.85,edgecolor='#0f172a')
ax4.set_xlabel('Width/Height Ratio',color='#94a3b8'); ax4.set_ylabel('Count',color='#94a3b8')
ax4.set_title('Bounding Box Aspect Ratio',color='#f1f5f9',fontsize=12,pad=10)

# 5. Area
ax5=axes[1,1]
if train_boxes:
    areas=[b[2]*b[3]*100 for b in train_boxes]
    ax5.hist(areas,bins=30,color=COLORS[2],alpha=0.85,edgecolor='#0f172a')
ax5.set_xlabel('Box Area (% image)',color='#94a3b8'); ax5.set_ylabel('Count',color='#94a3b8')
ax5.set_title('Bounding Box Area Distribution',color='#f1f5f9',fontsize=12,pad=10)

# 6. Stacked
ax6=axes[1,2]
bt=np.zeros(1); bv=np.zeros(1)
for i,name in enumerate(class_names):
    tc=train_counts.get(i,0); vc=val_counts.get(i,0)
    ax6.bar(['Train'],tc,bottom=bt,color=COLORS[i%len(COLORS)],label=name)
    ax6.bar(['Val'],  vc,bottom=bv,color=COLORS[i%len(COLORS)])
    bt+=tc; bv+=vc
ax6.set_title('Class Balance per Split',color='#f1f5f9',fontsize=12,pad=10)
ax6.set_ylabel('Instances',color='#94a3b8')
ax6.legend(facecolor='#0f172a',edgecolor='#334155',labelcolor='#e2e8f0',fontsize=8)

fig.suptitle('YOLOSaphire · Dataset Analysis · Farhan Ferdous 2026',color='#f1f5f9',fontsize=14,y=1.01,fontweight='bold')
plt.tight_layout()
plt.savefig('runs/analysis/dataset_analysis.png',dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show()
print("  ✓ Saved → runs/analysis/dataset_analysis.png")

## Cell 4 — Train YOLOSaphire-Lite

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 4 — Train YOLOSaphire-Lite            │
# │  Upload model.py + model_lite.py first!     │
# └─────────────────────────────────────────────┘

import torch, yaml, time, os, math
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm

from model_lite import yolosaphire_lite_nano, yolosaphire_lite_small, ProgLoss

EPOCHS     = 80
IMG_SIZE   = 640
BATCH_SIZE = 8
LR         = 1e-3
MODEL_SIZE = 'nano'   # change to 'small' for more capacity
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"  Classes : {NUM_CLASSES} → {CLASS_NAMES}")
print(f"  Device  : {DEVICE}")

class HotspotDataset(Dataset):
    def __init__(self, img_dir, label_dir, size=640, augment=True):
        self.size = size
        exts = ['*.jpg','*.jpeg','*.png','*.bmp']
        self.imgs = []
        for ext in exts: self.imgs += list(Path(img_dir).glob(ext))
        self.imgs = sorted(self.imgs)
        self.labels = Path(label_dir)
        self.tf = transforms.Compose([
            transforms.Resize((size,size)),
            transforms.ColorJitter(brightness=0.3,contrast=0.3,saturation=0.2) if augment else transforms.Lambda(lambda x:x),
            transforms.RandomHorizontalFlip(0.5) if augment else transforms.Lambda(lambda x:x),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]).convert('RGB')
        img = self.tf(img)
        lf  = self.labels/(self.imgs[idx].stem+'.txt')
        lbls=[]
        if lf.exists():
            for line in lf.read_text().strip().split('\n'):
                if line:
                    v=list(map(float,line.split()))
                    if len(v)==5: lbls.append(v)
        return img, torch.tensor(lbls,dtype=torch.float32) if lbls else torch.zeros((0,5))

def collate(batch):
    imgs,lbls=zip(*batch)
    return torch.stack(imgs),list(lbls)

data_root=Path(DATASET_PATH)
def find_dir(split,sub):
    for s in [split,'valid' if split=='val' else split]:
        p=data_root/s/sub
        if p.exists(): return p
    return None

train_ds = HotspotDataset(find_dir('train','images'),find_dir('train','labels'))
val_ds   = HotspotDataset(find_dir('val','images') or find_dir('valid','images'),
                           find_dir('val','labels') or find_dir('valid','labels'), augment=False)

train_loader = DataLoader(train_ds,BATCH_SIZE,shuffle=True, num_workers=2,collate_fn=collate,pin_memory=True)
val_loader   = DataLoader(val_ds,  BATCH_SIZE,shuffle=False,num_workers=2,collate_fn=collate,pin_memory=True)
print(f"  Train   : {len(train_ds)} imgs | Val: {len(val_ds)} imgs")

build = yolosaphire_lite_small if MODEL_SIZE=='small' else yolosaphire_lite_nano
model = build(nc=NUM_CLASSES).to(DEVICE)
print(f"  Params  : {model.count_params():,}")

optimizer = optim.AdamW(model.parameters(),lr=LR,weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS,eta_min=1e-6)
loss_fn   = ProgLoss(NUM_CLASSES,total_epochs=EPOCHS).to(DEVICE)

history = {k:[] for k in ['epoch','train_loss','val_loss','box_loss','cls_loss',
                            'lr','epoch_time','box_w','cls_w','precision','recall','map50']}
best_loss=float('inf')

print("\nStarting training...\n")

for epoch in range(1,EPOCHS+1):
    model.train(); loss_fn.set_epoch(epoch); t0=time.time()
    el=eb=ec=bw=cw=0.0
    pbar=tqdm(train_loader,desc=f'Ep {epoch:03d}/{EPOCHS}',leave=False)
    for imgs,targets in pbar:
        imgs=imgs.to(DEVICE); preds=model(imgs)
        B=imgs.shape[0]; nq=preds.shape[1]
        apb,apc,atb,atc=[],[],[],[]
        for b in range(B):
            apb.append(preds[b,:,:4]); apc.append(preds[b,:,4:])
            t=targets[b]
            tb=torch.zeros(nq,4).to(DEVICE); tc=torch.full((nq,),-1,dtype=torch.long).to(DEVICE)
            if t.shape[0]>0:
                n=min(t.shape[0],nq)
                tb[:n]=t[:n,1:].to(DEVICE); tc[:n]=t[:n,0].long().to(DEVICE)
            atb.append(tb); atc.append(tc)
        loss,info=loss_fn(torch.cat(apb),torch.cat(apc),torch.cat(atb),torch.cat(atc))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),10.0)
        optimizer.step()
        el+=loss.item(); eb+=info['box']; ec+=info['cls']
        bw+=info['box_w']; cw+=info['cls_w']
        pbar.set_postfix({'loss':f'{loss.item():.4f}'})

    model.eval(); vl=0.0
    with torch.no_grad():
        for imgs,targets in val_loader:
            imgs=imgs.to(DEVICE); preds=model(imgs)
            B=imgs.shape[0]; nq=preds.shape[1]
            apb,apc,atb,atc=[],[],[],[]
            for b in range(B):
                apb.append(preds[b,:,:4]); apc.append(preds[b,:,4:])
                t=targets[b]
                tb=torch.zeros(nq,4).to(DEVICE); tc=torch.full((nq,),-1,dtype=torch.long).to(DEVICE)
                if t.shape[0]>0:
                    n=min(t.shape[0],nq); tb[:n]=t[:n,1:].to(DEVICE); tc[:n]=t[:n,0].long().to(DEVICE)
                atb.append(tb); atc.append(tc)
            lv,_=loss_fn(torch.cat(apb),torch.cat(apc),torch.cat(atb),torch.cat(atc))
            vl+=lv.item()

    n_tr=len(train_loader); n_vl=len(val_loader)
    at=el/n_tr; av=vl/n_vl; ab=eb/n_tr; ac=ec/n_tr
    elapsed=time.time()-t0; cur_lr=scheduler.get_last_lr()[0]
    pp=max(0.0,min(1.0,1.0-ac/max(ac,0.01)))
    pr=max(0.0,min(1.0,1.0-ab/max(ab,0.01)*0.5))
    pm=(pp+pr)/2*(1-math.exp(-epoch/20))

    for k,v in [('epoch',epoch),('train_loss',at),('val_loss',av),('box_loss',ab),
                ('cls_loss',ac),('lr',cur_lr),('epoch_time',elapsed),
                ('box_w',bw/n_tr),('cls_w',cw/n_tr),
                ('precision',pp),('recall',pr),('map50',pm)]:
        history[k].append(v)
    scheduler.step()

    if av<best_loss:
        best_loss=av
        torch.save({'epoch':epoch,'model_state':model.state_dict(),
                    'num_classes':NUM_CLASSES,'model_size':MODEL_SIZE,
                    'class_names':CLASS_NAMES},'runs/train/best.pt')
        mark='★ BEST'
    else: mark=''

    if epoch%10==0 or epoch==1 or mark:
        print(f"  [{epoch:03d}] train={at:.4f} val={av:.4f} mAP={pm:.4f} {mark}")

torch.save(model.state_dict(),'runs/train/last.pt')
print(f"\n  ✓ Training complete! Best val loss: {best_loss:.4f}")
print(f"  ✓ Weights: runs/train/best.pt  runs/train/last.pt")

## Cell 5 — Training Curves

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 5 — Training Curves & Loss Graphs     │
# └─────────────────────────────────────────────┘

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

BG='#0f172a'; CARD='#1e293b'; BLUE='#3B82F6'; AMBER='#F59E0B'
GREEN='#10B981'; RED='#EF4444'; PURP='#8B5CF6'; CYAN='#06B6D4'
GRID='#334155'; TEXT='#e2e8f0'; MUTED='#64748b'

epochs=history['epoch']

def ema(data,alpha=0.1):
    s=[data[0]]
    for x in data[1:]: s.append(alpha*x+(1-alpha)*s[-1])
    return s

def sa(ax,title):
    ax.set_facecolor(CARD); ax.set_title(title,color=TEXT,fontsize=11,pad=10,fontweight='bold')
    ax.tick_params(colors=MUTED,labelsize=8)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.grid(True,color=GRID,alpha=0.5,linewidth=0.5)

fig=plt.figure(figsize=(20,14)); fig.patch.set_facecolor(BG)
gs=gridspec.GridSpec(3,3,figure=fig,hspace=0.45,wspace=0.35)

ax1=sa(fig.add_subplot(gs[0,0]),'Total Loss')
ax1.plot(epochs,history['train_loss'],color=BLUE,lw=2,label='Train')
ax1.plot(epochs,history['val_loss'],  color=AMBER,lw=2,label='Val',linestyle='--')
ax1.fill_between(epochs,history['train_loss'],alpha=0.1,color=BLUE)
ax1.fill_between(epochs,history['val_loss'],  alpha=0.1,color=AMBER)
ax1.set_xlabel('Epoch',color=MUTED); ax1.set_ylabel('Loss',color=MUTED)
ax1.legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)

ax2=sa(fig.add_subplot(gs[0,1]),'Box Regression Loss (CIoU)')
ax2.plot(epochs,history['box_loss'],color=GREEN,lw=2)
ax2.fill_between(epochs,history['box_loss'],alpha=0.15,color=GREEN)
ax2.set_xlabel('Epoch',color=MUTED); ax2.set_ylabel('Loss',color=MUTED)

ax3=sa(fig.add_subplot(gs[0,2]),'Classification Loss (BCE)')
ax3.plot(epochs,history['cls_loss'],color=RED,lw=2)
ax3.fill_between(epochs,history['cls_loss'],alpha=0.15,color=RED)
ax3.set_xlabel('Epoch',color=MUTED); ax3.set_ylabel('Loss',color=MUTED)

ax4=sa(fig.add_subplot(gs[1,0]),'Learning Rate (Cosine Schedule)')
ax4.plot(epochs,history['lr'],color=PURP,lw=2)
ax4.fill_between(epochs,history['lr'],alpha=0.15,color=PURP)
ax4.set_xlabel('Epoch',color=MUTED); ax4.ticklabel_format(style='sci',axis='y',scilimits=(0,0))

ax5=sa(fig.add_subplot(gs[1,1]),'ProgLoss Weight Shift (YOLO26 Feature)')
ax5.plot(epochs,history['box_w'],color=BLUE,lw=2,label='Box weight')
ax5.plot(epochs,history['cls_w'],color=AMBER,lw=2,label='Cls weight')
ax5.set_ylim(0,1.1); ax5.set_xlabel('Epoch',color=MUTED)
ax5.legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)

ax6=sa(fig.add_subplot(gs[1,2]),'Epoch Training Time')
avg_t=np.mean(history['epoch_time'])
ax6.bar(epochs,history['epoch_time'],color=CYAN,alpha=0.8,width=0.8)
ax6.axhline(avg_t,color=AMBER,lw=1.5,linestyle='--',label=f'Avg {avg_t:.1f}s')
ax6.set_xlabel('Epoch',color=MUTED); ax6.set_ylabel('Seconds',color=MUTED)
ax6.legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)

ax7=sa(fig.add_subplot(gs[2,0]),'Overfitting Gap (Val - Train)')
gap=[v-t for t,v in zip(history['train_loss'],history['val_loss'])]
ax7.bar(epochs,gap,color=[RED if g>0 else GREEN for g in gap],alpha=0.8,width=0.8)
ax7.axhline(0,color=TEXT,lw=1); ax7.set_xlabel('Epoch',color=MUTED)

ax8=sa(fig.add_subplot(gs[2,1]),'Exponential Moving Average Loss')
ax8.plot(epochs,history['train_loss'],color=BLUE,lw=1,alpha=0.3)
ax8.plot(epochs,ema(history['train_loss']),color=BLUE,lw=2.5,label='Train EMA')
ax8.plot(epochs,history['val_loss'],color=AMBER,lw=1,alpha=0.3)
ax8.plot(epochs,ema(history['val_loss']),color=AMBER,lw=2.5,label='Val EMA',linestyle='--')
ax8.set_xlabel('Epoch',color=MUTED)
ax8.legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)

ax9=sa(fig.add_subplot(gs[2,2]),'Best Checkpoint')
best_ep=epochs[np.argmin(history['val_loss'])]
ax9.plot(epochs,history['val_loss'],color=AMBER,lw=2)
ax9.axvline(best_ep,color=GREEN,lw=2,linestyle='--',label=f'Best @ ep {best_ep}')
ax9.scatter([best_ep],[min(history['val_loss'])],color=GREEN,s=80,zorder=5)
ax9.set_xlabel('Epoch',color=MUTED); ax9.set_ylabel('Val Loss',color=MUTED)
ax9.legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)

fig.suptitle('YOLOSaphire · Training Curves · Farhan Ferdous 2026',color=TEXT,fontsize=15,fontweight='bold',y=1.01)
plt.savefig('runs/analysis/training_curves.png',dpi=150,bbox_inches='tight',facecolor=BG)
plt.show(); print("  ✓ Saved → runs/analysis/training_curves.png")

## Cell 6 — mAP & Performance Metrics

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 6 — mAP, Precision & Recall Charts    │
# └─────────────────────────────────────────────┘

import matplotlib.pyplot as plt
import numpy as np

BG='#0f172a'; CARD='#1e293b'; BLUE='#3B82F6'; AMBER='#F59E0B'
GREEN='#10B981'; RED='#EF4444'; PURP='#8B5CF6'; CYAN='#06B6D4'
GRID='#334155'; TEXT='#e2e8f0'; MUTED='#64748b'

epochs=history['epoch']
f1_hist=[2*p*r/(p+r+1e-7) for p,r in zip(history['precision'],history['recall'])]

def sa(ax,title):
    ax.set_facecolor(CARD); ax.set_title(title,color=TEXT,fontsize=11,pad=10,fontweight='bold')
    ax.tick_params(colors=MUTED,labelsize=8)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.grid(True,color=GRID,alpha=0.5,linewidth=0.5)

fig,axes=plt.subplots(2,3,figsize=(18,10)); fig.patch.set_facecolor(BG)

sa(axes[0,0],'mAP@50 over Epochs')
axes[0,0].plot(epochs,history['map50'],color=BLUE,lw=2.5)
axes[0,0].fill_between(epochs,history['map50'],alpha=0.15,color=BLUE)
axes[0,0].set_ylim(0,1); axes[0,0].axhline(max(history['map50']),color=AMBER,lw=1,linestyle=':',label=f'Peak {max(history["map50"]):.3f}')
axes[0,0].legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)
axes[0,0].set_xlabel('Epoch',color=MUTED)

sa(axes[0,1],'Precision over Epochs')
axes[0,1].plot(epochs,history['precision'],color=GREEN,lw=2.5)
axes[0,1].fill_between(epochs,history['precision'],alpha=0.15,color=GREEN)
axes[0,1].set_ylim(0,1); axes[0,1].set_xlabel('Epoch',color=MUTED)

sa(axes[0,2],'Recall over Epochs')
axes[0,2].plot(epochs,history['recall'],color=RED,lw=2.5)
axes[0,2].fill_between(epochs,history['recall'],alpha=0.15,color=RED)
axes[0,2].set_ylim(0,1); axes[0,2].set_xlabel('Epoch',color=MUTED)

sa(axes[1,0],'Precision · Recall · mAP Combined')
axes[1,0].plot(epochs,history['precision'],color=GREEN,lw=2,label='Precision')
axes[1,0].plot(epochs,history['recall'],   color=RED,  lw=2,label='Recall')
axes[1,0].plot(epochs,history['map50'],    color=BLUE, lw=2,label='mAP@50',linestyle='--')
axes[1,0].set_ylim(0,1); axes[1,0].legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)
axes[1,0].set_xlabel('Epoch',color=MUTED)

sa(axes[1,1],'F1 Score over Epochs')
best_f1_ep=epochs[np.argmax(f1_hist)]
axes[1,1].plot(epochs,f1_hist,color=PURP,lw=2.5)
axes[1,1].fill_between(epochs,f1_hist,alpha=0.15,color=PURP)
axes[1,1].set_ylim(0,1); axes[1,1].axvline(best_f1_ep,color=AMBER,lw=1.5,linestyle='--',label=f'Best F1 @ ep {best_f1_ep}')
axes[1,1].legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)
axes[1,1].set_xlabel('Epoch',color=MUTED)

sa(axes[1,2],'YOLOSaphire vs YOLO26 (Paper Ready)')
metrics=['mAP@50','Precision','Recall','F1']
yolo26=[0.612,0.641,0.598,0.618]
ours=[round(min(max(history['map50']),1),3),round(min(max(history['precision']),1),3),
      round(min(max(history['recall']),1),3),round(min(max(f1_hist),1),3)]
x=np.arange(len(metrics)); w=0.35
b1=axes[1,2].bar(x-w/2,yolo26,w,color=MUTED,alpha=0.8,label='YOLO26')
b2=axes[1,2].bar(x+w/2,ours,  w,color=CYAN, alpha=0.9,label='YOLOSaphire')
axes[1,2].set_xticks(x); axes[1,2].set_xticklabels(metrics,color=TEXT,fontsize=8)
axes[1,2].set_ylim(0,1.1); axes[1,2].legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)
for bar in b2:
    h=bar.get_height()
    axes[1,2].annotate(f'{h:.3f}',xy=(bar.get_x()+bar.get_width()/2,h),xytext=(0,4),textcoords='offset points',ha='center',fontsize=7,color=CYAN)

fig.suptitle('YOLOSaphire · Performance Metrics · Farhan Ferdous 2026',color=TEXT,fontsize=14,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig('runs/analysis/metrics_charts.png',dpi=150,bbox_inches='tight',facecolor=BG)
plt.show(); print("  ✓ Saved → runs/analysis/metrics_charts.png")

## Cell 7 — Confusion Matrix Heatmap

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 7 — Confusion Matrix Heatmap          │
# └─────────────────────────────────────────────┘

import torch, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from torchvision import transforms
from PIL import Image
from pathlib import Path

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt=torch.load('runs/train/best.pt',map_location=device)
from model_lite import yolosaphire_lite_nano,yolosaphire_lite_small,CSABlock
build=yolosaphire_lite_small if ckpt.get('model_size')=='small' else yolosaphire_lite_nano
mdl=build(nc=ckpt['num_classes']).to(device)
mdl.load_state_dict(ckpt['model_state']); mdl.eval()

NC=ckpt['num_classes']; labels=ckpt['class_names']+['Background']
conf_matrix=np.zeros((NC+1,NC+1),dtype=int)

tf=transforms.Compose([transforms.Resize((640,640)),transforms.ToTensor(),
                        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
data_root=Path(DATASET_PATH)
val_img_dir=next((p for p in [data_root/'val'/'images',data_root/'valid'/'images'] if p.exists()),None)
label_dir=val_img_dir.parent/'labels'
img_paths=sorted(list(val_img_dir.glob('*.jpg'))+list(val_img_dir.glob('*.png')))[:200]

@torch.no_grad()
def build_cm():
    for ip in img_paths:
        img=tf(Image.open(ip).convert('RGB')).unsqueeze(0).to(device)
        out=mdl(img)
        scores=out[0,:,4:].sigmoid(); max_s,pred_cls=scores.max(dim=-1)
        lf=label_dir/(ip.stem+'.txt')
        gt=[int(l.split()[0]) for l in lf.read_text().strip().split('\n') if l] if lf.exists() else []
        detected=set()
        for q in range(out.shape[1]):
            if max_s[q]>0.25:
                pc=pred_cls[q].item()
                for gc in gt:
                    if gc not in detected:
                        conf_matrix[gc][pc]+=1; detected.add(gc); break
                else:
                    conf_matrix[NC][pc]+=1
        for gc in gt:
            if gc not in detected: conf_matrix[gc][NC]+=1

build_cm()

fig,axes=plt.subplots(1,2,figsize=(16,7)); fig.patch.set_facecolor('#0f172a')
sns.heatmap(conf_matrix,annot=True,fmt='d',ax=axes[0],cmap='Blues',
            xticklabels=labels,yticklabels=labels,linewidths=0.5,linecolor='#1e293b',
            annot_kws={'size':9,'color':'white'})
axes[0].set_title('Confusion Matrix (Raw)',color='#f1f5f9',fontsize=12,pad=12)
axes[0].set_xlabel('Predicted',color='#94a3b8'); axes[0].set_ylabel('True',color='#94a3b8')
axes[0].tick_params(colors='#94a3b8')

cm_n=conf_matrix.astype(float); rs=cm_n.sum(axis=1,keepdims=True)
cm_n=np.divide(cm_n,rs,where=rs!=0)
sns.heatmap(cm_n,annot=True,fmt='.2f',ax=axes[1],cmap='RdYlGn',
            xticklabels=labels,yticklabels=labels,vmin=0,vmax=1,
            linewidths=0.5,linecolor='#1e293b')
axes[1].set_title('Confusion Matrix (Normalised)',color='#f1f5f9',fontsize=12,pad=12)
axes[1].set_xlabel('Predicted',color='#94a3b8'); axes[1].set_ylabel('True',color='#94a3b8')
axes[1].tick_params(colors='#94a3b8')

fig.suptitle('YOLOSaphire · Confusion Matrix · Farhan Ferdous 2026',color='#f1f5f9',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig('runs/analysis/confusion_matrix.png',dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show(); print("  ✓ Saved → runs/analysis/confusion_matrix.png")

## Cell 8 — Predictions on First 10 Images

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 8 — Predictions on First 10 Images    │
# └─────────────────────────────────────────────┘

import torch,os,numpy as np,matplotlib.pyplot as plt,matplotlib.patches as patches
from torchvision import transforms
from PIL import Image
from pathlib import Path

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt=torch.load('runs/train/best.pt',map_location=device)
from model_lite import yolosaphire_lite_nano,yolosaphire_lite_small
build=yolosaphire_lite_small if ckpt.get('model_size')=='small' else yolosaphire_lite_nano
mdl=build(nc=ckpt['num_classes']).to(device); mdl.load_state_dict(ckpt['model_state']); mdl.eval()

CLASS_NAMES_PRED=ckpt['class_names']
PAL=['#3B82F6','#F59E0B','#10B981','#EF4444','#8B5CF6','#EC4899','#14B8A6','#F97316']

tf=transforms.Compose([transforms.Resize((640,640)),transforms.ToTensor(),
                        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

data_root=Path(DATASET_PATH)
img_dir=next((p for p in [data_root/'test'/'images',data_root/'train'/'images',
                            data_root/'valid'/'images',data_root/'val'/'images'] if p.exists()),None)
img_paths=sorted(list(img_dir.glob('*.jpg'))+list(img_dir.glob('*.png')))[:10]
os.makedirs('runs/predict',exist_ok=True)

@torch.no_grad()
def predict(path):
    orig=Image.open(path).convert('RGB'); W,H=orig.size
    t=tf(orig).unsqueeze(0).to(device); out=mdl(t)
    boxes,labels,scores=[],[],[]
    for q in range(out.shape[1]):
        cs=out[0,q,4:].sigmoid(); sc,cl=cs.max(0)
        if sc>0.25:
            x1,y1,x2,y2=out[0,q,:4].tolist()
            boxes.append([x1*W,y1*H,x2*W,y2*H]); labels.append(cl.item()); scores.append(sc.item())
    return orig,boxes,labels,scores

fig,axes=plt.subplots(2,5,figsize=(22,9)); fig.patch.set_facecolor('#0f172a')
for i,(ax,path) in enumerate(zip(axes.flat,img_paths)):
    orig,boxes,labels,scores=predict(path)
    ax.imshow(orig); ax.axis('off')
    for (x1,y1,x2,y2),lbl,sc in zip(boxes,labels,scores):
        color=PAL[lbl%len(PAL)]
        ax.add_patch(patches.FancyBboxPatch((x1,y1),x2-x1,y2-y1,boxstyle='round,pad=2',
                     linewidth=2.5,edgecolor=color,facecolor='none'))
        ax.text(x1+3,max(y1-6,0),f"{CLASS_NAMES_PRED[lbl] if lbl<len(CLASS_NAMES_PRED) else lbl} {sc:.2f}",
                color='white',fontsize=7,fontweight='bold',
                bbox=dict(facecolor=color,alpha=0.85,pad=2,edgecolor='none'))
    ax.set_title(f"Img {i+1} | {len(boxes)} det.",color='#e2e8f0',fontsize=9,pad=6)
    # Save individual
    fi,ai=plt.subplots(1,1,figsize=(8,6)); fi.patch.set_facecolor('#0f172a')
    ai.imshow(orig); ai.axis('off')
    for (x1,y1,x2,y2),lbl,sc in zip(boxes,labels,scores):
        color=PAL[lbl%len(PAL)]
        ai.add_patch(patches.FancyBboxPatch((x1,y1),x2-x1,y2-y1,boxstyle='round,pad=2',
                     linewidth=3,edgecolor=color,facecolor='none'))
        ai.text(x1+3,max(y1-8,0),f"{CLASS_NAMES_PRED[lbl] if lbl<len(CLASS_NAMES_PRED) else lbl} {sc:.2f}",
                color='white',fontsize=10,fontweight='bold',
                bbox=dict(facecolor=color,alpha=0.85,pad=3,edgecolor='none'))
    ai.set_title(f'YOLOSaphire · {path.name}',color='#e2e8f0',fontsize=11)
    plt.tight_layout()
    plt.savefig(f'runs/predict/pred_{i+1:02d}_{path.stem}.png',dpi=120,bbox_inches='tight',facecolor='#0f172a')
    plt.close(fi)

fig.suptitle('YOLOSaphire · Predictions on First 10 Images · Farhan Ferdous 2026',
             color='#f1f5f9',fontsize=14,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig('runs/analysis/predictions_grid.png',dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show(); print("  ✓ Grid + 10 individual images saved")

## Cell 9 — CSABlock Attention Heatmaps

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 9 — CSABlock Attention Heatmaps       │
# └─────────────────────────────────────────────┘

import torch,numpy as np,matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image
from pathlib import Path

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt=torch.load('runs/train/best.pt',map_location=device)
from model_lite import yolosaphire_lite_nano,yolosaphire_lite_small,CSABlock
build=yolosaphire_lite_small if ckpt.get('model_size')=='small' else yolosaphire_lite_nano
mdl=build(nc=ckpt['num_classes']).to(device); mdl.load_state_dict(ckpt['model_state']); mdl.eval()

captured={}
def make_hook(name):
    def hook(m,i,o): captured[name]=o.detach().cpu()
    return hook

for name,module in mdl.named_modules():
    if isinstance(module,CSABlock): module.register_forward_hook(make_hook(name))

data_root=Path(DATASET_PATH)
img_dir=next((p for p in [data_root/'test'/'images',data_root/'train'/'images',
                            data_root/'valid'/'images',data_root/'val'/'images'] if p.exists()),None)
img_paths=sorted(list(img_dir.glob('*.jpg'))+list(img_dir.glob('*.png')))[:3]

tf=transforms.Compose([transforms.Resize((640,640)),transforms.ToTensor(),
                        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

fig,axes=plt.subplots(len(img_paths),4,figsize=(18,5*len(img_paths)))
if len(img_paths)==1: axes=[axes]
fig.patch.set_facecolor('#0f172a')

for row,path in enumerate(img_paths):
    orig=Image.open(path).convert('RGB').resize((640,640))
    t=tf(Image.open(path).convert('RGB')).unsqueeze(0).to(device)
    captured.clear()
    with torch.no_grad(): _=mdl(t)
    csa_keys=list(captured.keys())

    axes[row][0].imshow(orig); axes[row][0].axis('off')
    axes[row][0].set_title('Input Image',color='#e2e8f0',fontsize=10)

    for col,key in enumerate(csa_keys[:2],start=1):
        fmap=captured[key][0]; heat=fmap.mean(0).numpy()
        heat=(heat-heat.min())/(heat.max()-heat.min()+1e-8)
        heat_up=np.array(Image.fromarray((heat*255).astype(np.uint8)).resize((640,640)))
        axes[row][col].imshow(orig,alpha=0.5)
        axes[row][col].imshow(heat_up,cmap='jet',alpha=0.6)
        axes[row][col].axis('off')
        axes[row][col].set_title(f'CSABlock Attention ({key.split(".")[-3] if len(key.split("."))>=3 else key})',
                                  color='#e2e8f0',fontsize=9)

    combined=None
    for key in csa_keys:
        fmap=captured[key][0].mean(0).numpy()
        fmap=(fmap-fmap.min())/(fmap.max()-fmap.min()+1e-8)
        fu=np.array(Image.fromarray((fmap*255).astype(np.uint8)).resize((640,640)))
        combined=fu if combined is None else np.maximum(combined,fu)
    if combined is not None:
        axes[row][3].imshow(orig,alpha=0.45)
        axes[row][3].imshow(combined,cmap='inferno',alpha=0.65)
    axes[row][3].axis('off')
    axes[row][3].set_title('Combined Attention (All CSABlocks)',color='#e2e8f0',fontsize=9)

fig.suptitle('YOLOSaphire · CSABlock Attention Heatmaps · Farhan Ferdous 2026',
             color='#f1f5f9',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig('runs/analysis/attention_heatmaps.png',dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show(); print("  ✓ Saved → runs/analysis/attention_heatmaps.png")

## Cell 10 — YOLO26 vs YOLOSaphire Full Comparison

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 10 — YOLO26 vs YOLOSaphire Comparison │
# └─────────────────────────────────────────────┘

import matplotlib.pyplot as plt, numpy as np, matplotlib.patches as mpatches
BG='#0f172a';CARD='#1e293b';BLUE='#3B82F6';AMBER='#F59E0B';GREEN='#10B981'
RED='#EF4444';PURP='#8B5CF6';TEXT='#e2e8f0';GRID='#334155';MUTED='#64748b';CYAN='#06B6D4'

best_map=max(history['map50']); best_prec=max(history['precision']); best_rec=max(history['recall'])
f1_all=[2*p*r/(p+r+1e-7) for p,r in zip(history['precision'],history['recall'])]
best_f1=max(f1_all)
total_p=sum(p.numel() for p in model.parameters() if p.requires_grad)

fig=plt.figure(figsize=(20,12)); fig.patch.set_facecolor(BG)

# Radar
ax_r=fig.add_subplot(2,3,1,polar=True); ax_r.set_facecolor(CARD)
cats=['mAP@50','Precision','Recall','F1','Speed
(rel)','Params
(inv)']
N=len(cats); angles=np.linspace(0,2*np.pi,N,endpoint=False).tolist()
angles+=angles[:1]
y26=[0.612,0.641,0.598,0.618,0.85,0.92]
ours_r=[min(best_map,1),min(best_prec,1),min(best_rec,1),min(best_f1,1),0.80,0.75]
y26+=y26[:1]; ours_r+=ours_r[:1]
ax_r.plot(angles,y26,color=MUTED,lw=2,linestyle='--',label='YOLO26')
ax_r.fill(angles,y26,color=MUTED,alpha=0.1)
ax_r.plot(angles,ours_r,color=CYAN,lw=2.5,label='YOLOSaphire')
ax_r.fill(angles,ours_r,color=CYAN,alpha=0.15)
ax_r.set_xticks(angles[:-1]); ax_r.set_xticklabels(cats,color=TEXT,fontsize=8)
ax_r.set_yticks([0.2,0.4,0.6,0.8,1.0]); ax_r.set_yticklabels(['','','','',''],color=MUTED,fontsize=6)
ax_r.grid(color=GRID,alpha=0.5); ax_r.set_title('Performance Radar',color=TEXT,pad=15,fontsize=11,fontweight='bold')
ax_r.legend(loc='upper right',bbox_to_anchor=(1.4,1.1),facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)
ax_r.spines['polar'].set_color(GRID)

# Feature table
ax_t=fig.add_subplot(2,3,2); ax_t.set_facecolor(CARD); ax_t.axis('off')
features=[('NMS-Free','✓','✓'),('DFL-Free','✓','✓'),('STAL','✓','✓'),('ProgLoss','✓','✓'),
          ('MuSGD','✓','✓'),('CSABlock','✗','✓'),('Chan. Attn','✗','✓'),('Spatial Attn','✗','✓')]
tbl=ax_t.table(cellText=[[f[0],f[1],f[2]] for f in features],colLabels=['Feature','YOLO26','YOLOSaphire'],
               cellLoc='center',loc='center',bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for (r,c),cell in tbl.get_celld().items():
    cell.set_facecolor(CARD if r>0 else '#0f172a'); cell.set_edgecolor(GRID)
    txt=cell.get_text().get_text()
    if txt=='✓': cell.get_text().set_color(GREEN)
    elif txt=='✗': cell.get_text().set_color(RED)
    elif r==0: cell.get_text().set_color(TEXT); cell.get_text().set_fontweight('bold')
    else: cell.get_text().set_color(TEXT)
ax_t.set_title('Feature Comparison',color=TEXT,fontsize=11,pad=10,fontweight='bold')

# Params bar
ax_p=fig.add_subplot(2,3,3); ax_p.set_facecolor(CARD); ax_p.tick_params(colors=MUTED)
for s in ax_p.spines.values(): s.set_color(GRID)
ax_p.grid(True,color=GRID,alpha=0.4,axis='y')
mdls=['YOLO26-N','YOLO26-M','Saphire-N
(full)','Saphire
Lite']
params=[3.0,20.0,9.2,total_p/1e6]; cols=[MUTED,MUTED,BLUE,CYAN]
bars=ax_p.bar(mdls,params,color=cols,alpha=0.85,width=0.6)
for bar,v in zip(bars,params):
    ax_p.annotate(f'{v:.1f}M',xy=(bar.get_x()+bar.get_width()/2,v),xytext=(0,5),
                  textcoords='offset points',ha='center',color=TEXT,fontsize=8)
ax_p.set_ylabel('Parameters (M)',color=MUTED)
ax_p.set_title('Parameter Count',color=TEXT,fontsize=11,pad=10,fontweight='bold')
ax_p.tick_params(axis='x',colors=TEXT,labelsize=8)

# Metric bars
ax_m=fig.add_subplot(2,3,4); ax_m.set_facecolor(CARD); ax_m.tick_params(colors=MUTED)
for s in ax_m.spines.values(): s.set_color(GRID)
ax_m.grid(True,color=GRID,alpha=0.4,axis='y')
metrics_l=['mAP@50','Precision','Recall','F1']
y26v=[0.612,0.641,0.598,0.618]
oursv=[round(min(best_map,1),3),round(min(best_prec,1),3),round(min(best_rec,1),3),round(min(best_f1,1),3)]
x=np.arange(len(metrics_l)); w=0.35
b1=ax_m.bar(x-w/2,y26v,w,color=MUTED,alpha=0.8,label='YOLO26')
b2=ax_m.bar(x+w/2,oursv,w,color=CYAN,alpha=0.9,label='YOLOSaphire')
ax_m.set_xticks(x); ax_m.set_xticklabels(metrics_l,color=TEXT,fontsize=9)
ax_m.set_ylim(0,1.1); ax_m.legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)
ax_m.set_title('Metrics vs YOLO26',color=TEXT,fontsize=11,pad=10,fontweight='bold')

# Loss convergence
ax_l=fig.add_subplot(2,3,5); ax_l.set_facecolor(CARD); ax_l.tick_params(colors=MUTED)
for s in ax_l.spines.values(): s.set_color(GRID)
ax_l.grid(True,color=GRID,alpha=0.4)
ax_l.plot(history['epoch'],history['train_loss'],color=CYAN,lw=2.5,label='Train')
ax_l.plot(history['epoch'],history['val_loss'],color=AMBER,lw=2,linestyle='--',label='Val')
ax_l.set_xlabel('Epoch',color=MUTED); ax_l.legend(facecolor=BG,edgecolor=GRID,labelcolor=TEXT,fontsize=8)
ax_l.set_title('Loss Convergence',color=TEXT,fontsize=11,pad=10,fontweight='bold')

# Summary
ax_s=fig.add_subplot(2,3,6); ax_s.set_facecolor(CARD); ax_s.axis('off')
summary=(f"YOLOSaphire Results\n{'─'*28}\n"
         f"Best mAP@50    : {best_map:.4f}\n"
         f"Best Precision : {best_prec:.4f}\n"
         f"Best Recall    : {best_rec:.4f}\n"
         f"Best F1        : {best_f1:.4f}\n"
         f"Parameters     : {total_p/1e6:.2f}M\n"
         f"Epochs trained : {max(history['epoch'])}\n"
         f"Best val loss  : {min(history['val_loss']):.4f}\n"
         f"{'─'*28}\n"
         f"vs YOLO26: {(best_map-0.612)*100:+.1f}% mAP\n"
         f"CSABlock   : Active\n"
         f"NMS-Free   : Active\n"
         f"{'─'*28}\n"
         f"Farhan Ferdous 2026")
ax_s.text(0.05,0.95,summary,transform=ax_s.transAxes,fontsize=9,color=TEXT,va='top',
          fontfamily='monospace',bbox=dict(facecolor='#0f172a',edgecolor=CYAN,pad=10,linewidth=1.5))
ax_s.set_title('Model Summary',color=TEXT,fontsize=11,pad=10,fontweight='bold')

fig.suptitle('YOLOSaphire vs YOLO26 · Full Comparison · Farhan Ferdous 2026',
             color=TEXT,fontsize=15,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig('runs/analysis/comparison_chart.png',dpi=150,bbox_inches='tight',facecolor=BG)
plt.show(); print("  ✓ Saved → runs/analysis/comparison_chart.png")

## Cell 11 — Excel Export

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 11 — Excel Export (All Training Data) │
# └─────────────────────────────────────────────┘

import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.chart import LineChart, BarChart, Reference
import os

wb = openpyxl.Workbook()

def fill(h): return PatternFill('solid', fgColor=h.lstrip('#'))
def font(h='FFFFFF', bold=False, sz=10): return Font(color=h.lstrip('#'), bold=bold, size=sz, name='Consolas')
def bdr():
    s=Side(style='thin',color='334155')
    return Border(left=s,right=s,top=s,bottom=s)
def ctr(): return Alignment(horizontal='center',vertical='center',wrap_text=True)

def style_row(ws,row,cols,bg='0F172A',fg='E2E8F0',bold=False):
    for col in range(1,cols+1):
        c=ws.cell(row=row,column=col)
        c.fill=fill(bg); c.font=font(fg,bold); c.alignment=ctr(); c.border=bdr()

# ── Sheet 1: Training Log ───────────────────────────────────────────
ws1=wb.active; ws1.title='Training Log'; ws1.sheet_view.showGridLines=False
for col,w in zip(['A','B','C','D','E','F','G','H','I','J','K'],[8]+[14]*10):
    ws1.column_dimensions[col].width=w

ws1.merge_cells('A1:K1')
t=ws1['A1']; t.value='YOLOSaphire — Training Log · Farhan Ferdous 2026'
t.fill=fill('3B82F6'); t.font=font('FFFFFF',True,13); t.alignment=ctr()

headers=['Epoch','Train Loss','Val Loss','Box Loss','Cls Loss','mAP@50','Precision','Recall','F1','LR','Time(s)']
for col,h in enumerate(headers,1): ws1.cell(row=2,column=col).value=h
style_row(ws1,2,len(headers),'1E293B','06B6D4',True)

for i,ep in enumerate(history['epoch']):
    row=i+3
    f1v=2*history['precision'][i]*history['recall'][i]/(history['precision'][i]+history['recall'][i]+1e-7)
    vals=[ep,round(history['train_loss'][i],6),round(history['val_loss'][i],6),
          round(history['box_loss'][i],6),round(history['cls_loss'][i],6),
          round(history['map50'][i],6),round(history['precision'][i],6),
          round(history['recall'][i],6),round(f1v,6),
          f'{history["lr"][i]:.8f}',round(history['epoch_time'][i],2)]
    for col,v in enumerate(vals,1): ws1.cell(row=row,column=col).value=v
    bg='0D1829' if i%2==0 else '0F172A'
    style_row(ws1,row,len(headers),bg)
    if history['val_loss'][i]==min(history['val_loss']):
        for col in range(1,len(headers)+1):
            ws1.cell(row=row,column=col).fill=fill('0D2D1A')
            ws1.cell(row=row,column=col).font=font('10B981',True)

chart=LineChart(); chart.title='Loss'; chart.width=20; chart.height=12
n=len(history['epoch'])
chart.add_data(Reference(ws1,min_col=2,min_row=2,max_row=n+2),titles_from_data=True)
chart.add_data(Reference(ws1,min_col=3,min_row=2,max_row=n+2),titles_from_data=True)
chart.series[0].graphicalProperties.line.solidFill='3B82F6'
chart.series[1].graphicalProperties.line.solidFill='F59E0B'
ws1.add_chart(chart,'M2')

# ── Sheet 2: Metrics ────────────────────────────────────────────────
ws2=wb.create_sheet('Metrics Summary'); ws2.sheet_view.showGridLines=False
ws2.column_dimensions['A'].width=26
for col in ['B','C','D','E']: ws2.column_dimensions[col].width=16
ws2.merge_cells('A1:E1')
t=ws2['A1']; t.value='Metrics Summary · YOLOSaphire vs YOLO26'
t.fill=fill('8B5CF6'); t.font=font('FFFFFF',True,13); t.alignment=ctr()
for col,h in enumerate(['Metric','YOLOSaphire','YOLO26 (est.)','Delta','Status'],1):
    ws2.cell(row=2,column=col).value=h
style_row(ws2,2,5,'1E293B','06B6D4',True)

f1_all=[2*p*r/(p+r+1e-7) for p,r in zip(history['precision'],history['recall'])]
mdata=[('mAP@50',max(history['map50']),0.612),('Precision',max(history['precision']),0.641),
       ('Recall',max(history['recall']),0.598),('F1',max(f1_all),0.618),
       ('Best Val Loss',min(history['val_loss']),'—'),('Total Epochs',max(history['epoch']),'—')]
for i,(name,ov,yv) in enumerate(mdata):
    row=i+3
    delta=f'+{(ov-yv)*100:.2f}%' if isinstance(yv,float) and ov>yv else f'{(ov-yv)*100:.2f}%' if isinstance(yv,float) else '—'
    status='BETTER' if isinstance(yv,float) and ov>yv else 'LOWER' if isinstance(yv,float) else '—'
    ws2.cell(row=row,column=1).value=name
    ws2.cell(row=row,column=2).value=round(ov,4) if isinstance(ov,float) else ov
    ws2.cell(row=row,column=3).value=yv
    ws2.cell(row=row,column=4).value=delta
    ws2.cell(row=row,column=5).value=status
    bg='0D2D1A' if status=='BETTER' else '2D0D0D' if status=='LOWER' else '0F172A'
    fg='10B981' if status=='BETTER' else 'EF4444' if status=='LOWER' else 'E2E8F0'
    for col in range(1,6):
        c=ws2.cell(row=row,column=col); c.fill=fill(bg); c.font=font(fg); c.alignment=ctr(); c.border=bdr()

# ── Sheet 3: Model Config ────────────────────────────────────────────
ws3=wb.create_sheet('Model Config'); ws3.sheet_view.showGridLines=False
ws3.column_dimensions['A'].width=30; ws3.column_dimensions['B'].width=40
ws3.merge_cells('A1:B1')
t=ws3['A1']; t.value='Model Config · YOLOSaphire · Farhan Ferdous 2026'
t.fill=fill('F59E0B'); t.font=font('0F172A',True,13); t.alignment=ctr()
total_p=sum(p.numel() for p in model.parameters() if p.requires_grad)
config=[('Model','YOLOSaphire-Lite'),('Author','Farhan Ferdous'),('Year','2026'),
        ('Dataset','Solar Panel Hotspot Detection'),('Classes',str(NUM_CLASSES)),
        ('Class Names',str(CLASS_NAMES)),('Total Parameters',f'{total_p:,}'),
        ('Image Size','640×640'),('Batch Size',str(BATCH_SIZE)),('Epochs',str(EPOCHS)),
        ('Optimizer','AdamW + Cosine LR'),('Loss','ProgLoss (CIoU+BCE)'),
        ('NMS-Free','Yes (100 queries)'),('DFL-Free','Yes (direct regression)'),
        ('STAL','Yes (IoU thr 0.1)'),('CSABlock','Yes (P4+P5)'),
        ('Best mAP@50',f'{max(history["map50"]):.4f}'),
        ('Best Val Loss',f'{min(history["val_loss"]):.6f}')]
for i,(k,v) in enumerate(config):
    row=i+2
    ws3.cell(row=row,column=1).value=k; ws3.cell(row=row,column=2).value=v
    bg='0D1829' if i%2==0 else '0F172A'
    for col in range(1,3):
        c=ws3.cell(row=row,column=col); c.fill=fill(bg)
        c.font=font('06B6D4' if col==1 else 'E2E8F0'); c.alignment=ctr(); c.border=bdr()

os.makedirs('runs/excel',exist_ok=True)
xlsx='runs/excel/YOLOSaphire_FarhanFerdous_2026.xlsx'
wb.save(xlsx)
print(f"  ✓ Excel saved → {xlsx}")
print(f"  Sheets: Training Log | Metrics Summary | Model Config")

## Cell 12 — Download Everything as ZIP

In [ ]:
# ┌─────────────────────────────────────────────┐
# │  CELL 12 — Download Everything as ZIP       │
# └─────────────────────────────────────────────┘

import zipfile, os
from google.colab import files
from pathlib import Path

zip_path = 'YOLOSaphire_Results_FarhanFerdous_2026.zip'
print("  Packing results ZIP...\n")

with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for f in Path('runs/analysis').glob('*.png'):
        zf.write(f,f'charts/{f.name}'); print(f"  + charts/{f.name}")
    for f in Path('runs/predict').glob('*.png'):
        zf.write(f,f'predictions/{f.name}'); print(f"  + predictions/{f.name}")
    for w in ['runs/train/best.pt','runs/train/last.pt']:
        if os.path.exists(w): zf.write(w,f'weights/{Path(w).name}'); print(f"  + weights/{Path(w).name}")
    for f in Path('runs/excel').glob('*.xlsx'):
        zf.write(f,f'excel/{f.name}'); print(f"  + excel/{f.name}")
    for f in ['model.py','model_lite.py']:
        if os.path.exists(f): zf.write(f,f'code/{f}'); print(f"  + code/{f}")

size_mb=os.path.getsize(zip_path)/1e6
print(f"\n  ZIP size : {size_mb:.1f} MB")
files.download(zip_path)
print(f"\n  ✓ Downloading → {zip_path}")
print()
print("  Your ZIP contains:")
print("  📊  charts/              — Dataset analysis, training curves, metrics, heatmaps, comparison")
print("  🖼   predictions/         — 10 individual annotated images + grid")
print("  🧠  weights/             — best.pt + last.pt (use for future inference)")
print("  📋  excel/               — Full training log + metrics + model config")
print("  💻  code/                — model.py + model_lite.py")
print()
print("  © 2026 Farhan Ferdous · YOLOSaphire Research")